# RKD Distillation (Colab GPU)

This notebook runs student distillation with teacher checkpoints on Colab GPU.

In [ ]:
# Optional: mount Google Drive for persistent checkpoints
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")

In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def run(cmd):
    print("\n>>>", " ".join(cmd))
    subprocess.run(cmd, check=True)


REPO_URL = "https://github.com/Gabomfim/MO434.git"
REPO_ROOT = Path("/content/MO434")
RKD_ROOT = REPO_ROOT / "RKD"

if not REPO_ROOT.exists():
    run(["git", "clone", REPO_URL, str(REPO_ROOT)])
else:
    run(["git", "-C", str(REPO_ROOT), "pull"])

os.chdir(RKD_ROOT)
print("Working directory:", os.getcwd())
run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "tqdm",
        "h5py",
        "scipy",
        "wandb",
        "kagglehub",
    ]
)

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is required. In Colab, go to Runtime > Change runtime type > GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA version:", torch.version.cuda)

## Configure Distillation Run

In [ ]:
# Core config
DATASET = "cub200"
DATA_DIR = "../data"

# Student
BASE = "resnet18"
EMBEDDING_SIZE = "64"
L2NORMALIZE = "false"

# Teacher
TEACHER_BASE = "resnet50"
TEACHER_EMBEDDING_SIZE = "512"
TEACHER_L2NORMALIZE = "true"
TEACHER_LOAD = "teacher/best.pth"  # update if your checkpoint is elsewhere

# Distillation losses
TRIPLET_RATIO = "0"
DIST_RATIO = "1"
ANGLE_RATIO = "2"
QUAD_RATIO = "0"
DARK_RATIO = "0"
DARK_ALPHA = "2"
DARK_BETA = "3"
AT_RATIO = "0"

# Optimization
LR = "1e-4"
EPOCHS = "80"
BATCH = "128"
ITER_PER_EPOCH = "100"
LR_DECAY_EPOCHS = ["40", "60"]
LR_DECAY_GAMMA = "0.1"
SAVE_DIR = "student"

# W&B
WANDB_PROJECT = "rkd-metric-learning"
WANDB_ENTITY = ""  # optional
WANDB_RUN_NAME = "distill-resnet18-colab"
WANDB_MODE = "online"  # online | offline | disabled

In [ ]:
from pathlib import Path
import kagglehub

if DATASET == "cub200":
    path = Path(kagglehub.dataset_download("wenewone/cub2002011"))
    data_root = path.parent if path.name == "CUB_200_2011" else path
    DATA_DIR = str(data_root)
    print("Path to dataset files:", str(path))
    print("Using --data:", DATA_DIR)
else:
    print(
        f"Skipping Kaggle download because DATASET={DATASET}. Using --data: {DATA_DIR}"
    )

In [ ]:
import run_distill as distill_runner

distill_params = {
    "dataset": DATASET,
    "base": BASE,
    "teacher_base": TEACHER_BASE,
    "triplet_ratio": TRIPLET_RATIO,
    "dist_ratio": DIST_RATIO,
    "angle_ratio": ANGLE_RATIO,
    "quad_ratio": QUAD_RATIO,
    "dark_ratio": DARK_RATIO,
    "dark_alpha": DARK_ALPHA,
    "dark_beta": DARK_BETA,
    "at_ratio": AT_RATIO,
    "triplet_sample": "distance",
    "triplet_margin": "0.2",
    "l2normalize": L2NORMALIZE,
    "embedding_size": EMBEDDING_SIZE,
    "teacher_load": TEACHER_LOAD,
    "teacher_l2normalize": TEACHER_L2NORMALIZE,
    "teacher_embedding_size": TEACHER_EMBEDDING_SIZE,
    "lr": LR,
    "data": DATA_DIR,
    "epochs": EPOCHS,
    "batch": BATCH,
    "iter_per_epoch": ITER_PER_EPOCH,
    "lr_decay_epochs": LR_DECAY_EPOCHS,
    "lr_decay_gamma": LR_DECAY_GAMMA,
    "save_dir": SAVE_DIR,
    "wandb_project": WANDB_PROJECT,
    "wandb_run_name": WANDB_RUN_NAME,
    "wandb_mode": WANDB_MODE,
}

if WANDB_ENTITY.strip():
    distill_params["wandb_entity"] = WANDB_ENTITY

distill_runner.run_with_params(distill_params)

In [ ]:
import run as teacher_runner

# Evaluate student after distillation
student_eval_params = {
    "mode": "eval",
    "dataset": DATASET,
    "base": BASE,
    "embedding_size": EMBEDDING_SIZE,
    "l2normalize": L2NORMALIZE,
    "batch": BATCH,
    "recall": ["1"],
    "load": f"{SAVE_DIR}/best.pth",
    "data": DATA_DIR,
    "wandb_project": WANDB_PROJECT,
    "wandb_run_name": WANDB_RUN_NAME + "-eval",
    "wandb_mode": WANDB_MODE,
}

if WANDB_ENTITY.strip():
    student_eval_params["wandb_entity"] = WANDB_ENTITY

teacher_runner.run_with_params(student_eval_params)